In [3]:
import heapq

class Node:
    def __init__(self, board, parent=None, move="", g=0, h=0):
        self.board = board
        self.parent = parent
        self.move = move
        self.g = g  # Cost from start to current node
        self.h = h  # Heuristic cost (Manhattan Distance)
        self.f = g + h  # Total cost

    # Define comparison operators for priority queue ordered by f(n)
    def __lt__(self, other):
        return self.f < other.f

    def get_blank_pos(self):
        for r in range(3):
            for c in range(3):
                if self.board[r][c] == 0:
                    return r, c

def calculate_manhattan(board, goal):
    distance = 0
    goal_positions = {}
    for r in range(3):
        for c in range(3):
            goal_positions[goal[r][c]] = (r, c)

    for r in range(3):
        for c in range(3):
            val = board[r][c]
            if val != 0:
                target_r, target_c = goal_positions[val]
                distance += abs(r - target_r) + abs(c - target_c)
    return distance

def get_neighbors(node, goal):
    neighbors = []
    r, c = node.get_blank_pos()
    moves = {
        "UP": (r - 1, c),
        "DOWN": (r + 1, c),
        "LEFT": (r, c - 1),
        "RIGHT": (r, c + 1)
    }

    for move_name, (nr, nc) in moves.items():
        if 0 <= nr < 3 and 0 <= nc < 3:
            # Deep copy of the board
            new_board = [row[:] for row in node.board]
            # Swap blank space
            new_board[r][c], new_board[nr][nc] = new_board[nr][nc], new_board[r][c]

            h = calculate_manhattan(new_board, goal)
            neighbors.append(Node(new_board, parent=node, move=move_name, g=node.g + 1, h=h))
    return neighbors

def solve_8_puzzle(initial_board, goal_board):
    initial_h = calculate_manhattan(initial_board, goal_board)
    start_node = Node(initial_board, g=0, h=initial_h)

    open_set = []
    heapq.heappush(open_set, start_node)

    # Store string representation of board to check for visited states
    closed_set = set()

    while open_set:
        current_node = heapq.heappop(open_set)

        board_str = str(current_node.board)
        if board_str in closed_set:
            continue
        closed_set.add(board_str)

        if current_node.h == 0:
            # Reconstruct the path
            path = []
            curr = current_node
            while curr:
                path.append(curr)
                curr = curr.parent
            return path[::-1]

        for neighbor in get_neighbors(current_node, goal_board):
            if str(neighbor.board) not in closed_set:
                heapq.heappush(open_set, neighbor)

    return None

def print_board(board):
    for row in board:
        print(" ".join(map(str, row)))
    print()

def parse_input(prompt):
    print(prompt)
    user_in = input().strip().split()
    numbers = [int(x) for x in user_in]
    if len(numbers) != 9:
        raise ValueError("Please enter exactly 9 numbers representing a 3x3 board.")
    return [numbers[i:i+3] for i in range(0, 9, 3)]

try:
    # Get custom dynamic inputs
    print("--- 8-Puzzle A* Solver Configuration ---")
    initial = parse_input("Enter initial board state (9 space-separated values, 0 for empty): ")
    goal = parse_input("Enter goal board state (9 space-separated values, 0 for empty): ")

    print("\nInitial state configured:")
    print_board(initial)
    print("Goal state configured:")
    print_board(goal)

    solution_path = solve_8_puzzle(initial, goal)
    if solution_path:
        print(f"Optimal Solution Found in {len(solution_path) - 1} moves:\n")
        for step, node in enumerate(solution_path):
            if step > 0:
                print(f"Move: {node.move}")
            print_board(node.board)
    else:
        print("No solution found.")
except Exception as e:
    print(f"Error: {e}")


--- 8-Puzzle A* Solver Configuration ---
Enter initial board state (9 space-separated values, 0 for empty): 
 2 8 3 1 6 4 7 0 5
Enter goal board state (9 space-separated values, 0 for empty): 
 1 2 3 8 0 4 7 6 5

Initial state configured:
2 8 3
1 6 4
7 0 5

Goal state configured:
1 2 3
8 0 4
7 6 5

Optimal Solution Found in 5 moves:

2 8 3
1 6 4
7 0 5

Move: UP
2 8 3
1 0 4
7 6 5

Move: UP
2 0 3
1 8 4
7 6 5

Move: LEFT
0 2 3
1 8 4
7 6 5

Move: DOWN
1 2 3
0 8 4
7 6 5

Move: RIGHT
1 2 3
8 0 4
7 6 5

